# Step 4 — VLM feature extraction

**Prerequisites:** Renders from step 2 (`outputs/notebooks/02_rendering/rgb_*.npy`) or re-render inline with **`02_rendering_mesh_debug.ipynb`**.

**Pass checklist:**
- Patch tensor shape `(N_patches, D)` matches model grid (7×7 for CLIP-B/32 @ 224px)
- Text embeddings are L2-normalized; related verbs align better than unrelated ones (text–text)
- Global image embedding yields non-flat similarities across verbs (not a hard grasp test on arbitrary meshes)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from datasets.mesh_loading import load_mesh, normalize_mesh
from rendering.renderer import render_mesh_views
from utils.config import load_config, resolve_path
from vlm.feature_cache import save_patch_features, save_text_embeddings
from vlm.patch_extractor import encode_image_features, extract_patch_features
from vlm.text_encoder import cosine_similarity, encode_texts
from vlm.vlm_wrapper import VLMWrapper, build_vlm_config

In [ ]:
cfg = load_config()
render_cache = ROOT / "outputs" / "notebooks" / "02_rendering"
vlm_cache = ROOT / "outputs" / "notebooks" / "04_vlm"
vlm_cache.mkdir(parents=True, exist_ok=True)

rgb_paths = sorted(render_cache.glob("rgb_*.npy"))
if rgb_paths:
    rgbs = [np.load(p) for p in rgb_paths]
    print(f"loaded {len(rgbs)} cached renders from {render_cache}")
else:
    print("no render cache — re-rendering mesh views")
    mesh = normalize_mesh(load_mesh(resolve_path(cfg["rendering"]["mesh_path"])))
    views = render_mesh_views(mesh, cfg)
    rgbs = [v.rgb for v in views]

vlm_cfg = build_vlm_config(cfg)
print("VLM:", vlm_cfg.model_name)
wrapper = VLMWrapper(vlm_cfg)

In [ ]:
patch_list = extract_patch_features(wrapper, rgbs)
pf = patch_list[0]
print(f"patches {tuple(pf.patches.shape)} grid {pf.grid_h}x{pf.grid_w} dim {pf.feature_dim}")
print(f"expected num_patches={wrapper.num_patches}")

for i, feats in enumerate(patch_list):
    save_patch_features(feats, vlm_cache / f"patches_{i}.pt")

In [ ]:
# Patch norm heatmap on first view (mean token L2 per cell)
grid = pf.patches.norm(dim=-1).reshape(pf.grid_h, pf.grid_w).numpy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(rgbs[0])
axes[0].set_title("render view 0")
axes[0].axis("off")
im = axes[1].imshow(grid, cmap="magma")
axes[1].set_title("patch L2 norm (7x7)")
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
verbs = cfg["vlm"].get("sample_verbs", ["grasp", "sit on", "open", "pour from"])
text_emb = encode_texts(wrapper, verbs)
save_text_embeddings(text_emb, verbs, vlm_cache / "text_embeddings.pt")

norms = text_emb.norm(dim=-1)
print("text norms:", norms.tolist())

sim = cosine_similarity(text_emb, text_emb)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(sim.numpy(), vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(verbs)), verbs, rotation=45, ha="right")
ax.set_yticks(range(len(verbs)), verbs)
ax.set_title("verb–verb cosine similarity")
plt.tight_layout()
plt.show()

In [ ]:
# Global CLIP image embedding vs verbs (sanity check; patches used later for projection)
image_emb = encode_image_features(wrapper, [rgbs[0]])
img_text_sim = cosine_similarity(image_emb, text_emb).squeeze(0)
print("view0 image-text sim:", dict(zip(verbs, [round(x, 4) for x in img_text_sim.tolist()])))
print("highest verb:", verbs[img_text_sim.argmax().item()])

In [ ]:
assert pf.num_patches == wrapper.num_patches
assert pf.patches.shape[-1] == wrapper.feature_dim
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-4)
assert torch.isfinite(pf.patches).all()
assert pf.patches.std() > 1e-3, "patch features collapsed"

sim_spread = img_text_sim.max() - img_text_sim.min()
assert sim_spread.item() > 0.01, f"image-text scores too flat: {sim_spread.item():.4f}"

text_probe = encode_texts(wrapper, ["grasp", "hold something", "airplane in the sky"])
tt = cosine_similarity(text_probe, text_probe)
assert tt[0, 1].item() > tt[0, 2].item(), "text encoder: related verbs should align better"

unrelated = encode_texts(wrapper, ["airplane in the sky"])
sim_unrelated = cosine_similarity(image_emb, unrelated).item()
sim_best_verb = img_text_sim.max().item()
if sim_unrelated >= sim_best_verb:
    print(
        "NOTE: unrelated caption beat all sample verbs on this mesh — OK for arbitrary "
        "sample.glb; projection step uses per-patch features, not this global score."
    )

print("PASS: Step 4 VLM features")